# Domain E: Functional Connectivity and Circuit Interactions

This notebook addresses five questions:
- **E1:** Do noise correlations reveal cell-type-specific connectivity motifs?
- **E2:** Can transfer entropy / Granger causality reveal directed functional interactions?
- **E3:** How does population coupling relate to cell type and spatial position?
- **E4:** Does spontaneous activity structure during grey-screen epochs reveal cell-type-specific network dynamics? *(Zarr data)*
- **E5:** Do catch-trial responses reveal cell-type-specific expectation signals? *(Zarr data)*

**Data:** Zarr multimodal stores with ΔF/F traces (cells × trials), 3D coordinates, cell-type labels, gene expression, spontaneous activity, catch trials, and GLM data.

**Note:** E2 (Granger causality) requires continuous time-series data. The current dataset stores trial-level responses. Sections requiring continuous ΔF/F traces are marked `# ⚠️ REQUIRES: continuous time-series data`.

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import kruskal, spearmanr, pearsonr, mannwhitneyu
from scipy.spatial.distance import cdist, pdist, squareform
import warnings
warnings.filterwarnings('ignore')

from functions.data_loading import load_mouse_zarr, load_zarr_10hz
from functions.analysis import xcorr_pair, pairwise_granger

sns.set_context('talk')
sns.set_style('whitegrid')

# ══════════════════════════════════════════════════════════════════════
# Load data from zarr multimodal stores
# ══════════════════════════════════════════════════════════════════════
ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

adata_list = [load_mouse_zarr(mid, zarr_dir=ZARR_DIR, include_genes=True) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

coords = obs[['x_loc', 'y_loc', 'z_loc']].values.astype(float)
orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])

# Gene columns
META_COLS = {'unique_id', 'mouse_id', 'class_name', 'subclass_name', 'subclass_label',
             'supertype_name', 'cluster_name', 'cluster_alias', 'x_loc', 'y_loc', 'z_loc'}
GENE_COLS = [c for c in obs.columns if c not in META_COLS]

print(f"Total cells: {len(obs)}, Trials: {X.shape[1]}, Genes: {len(GENE_COLS)}")

## E5: Catch-Trial Expectation Signals

Analyse catch (blank) trials interleaved among gratings to detect cell-type-specific expectation or omission signals. 27 catch trials per session, 51 timepoints (−1 to +4 s).

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E4  Spontaneous Activity Assemblies from Grey-Screen Data (Zarr)
# ══════════════════════════════════════════════════════════════════════
import zarr
from sklearn.decomposition import NMF

ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']

# ── Load spontaneous activity from zarr ──
assembly_results = {}
for mouse_id in MOUSE_IDS:
    z = zarr.open(f'{ZARR_DIR}/{mouse_id}_multimodal_data.zarr', 'r')
    
    # Use session_1, long grey-screen epoch (index 1, ~360 s)
    gs = z['ophys/drifting_gratings/session_1/stim_aligned_dff/GreyScreen']
    dff_spont = gs['dff'][1]          # (3624, n_cells)
    running_spont = gs['running'][1]  # (3624, 2) — speed, acceleration
    time_spont = gs['time_relative'][:]
    
    # Cell-type labels
    subclass_names = z['transcriptomics/cell_type/subclass_name'][:]
    n_cells = dff_spont.shape[1]
    
    # Clip negative values for NMF (shift to non-negative)
    dff_shifted = dff_spont - dff_spont.min(axis=0, keepdims=True)
    
    # NMF assembly detection
    n_assemblies = 10
    nmf = NMF(n_components=n_assemblies, init='nndsvda', random_state=42, max_iter=500)
    W = nmf.fit_transform(dff_shifted)  # (timepoints, n_assemblies) — activation timecourses
    H = nmf.components_                  # (n_assemblies, n_cells) — cell weights
    
    # Assembly cell-type composition
    composition = {}
    for a in range(n_assemblies):
        top_cells = np.argsort(H[a])[-int(n_cells * 0.1):]  # top 10% contributing cells
        types_in_assembly = subclass_names[top_cells]
        unique, counts = np.unique(types_in_assembly, return_counts=True)
        composition[a] = dict(zip(unique, counts))
    
    # Running–assembly correlation
    run_speed = running_spont[:, 0]
    assembly_run_corr = np.array([pearsonr(W[:, a], run_speed)[0] for a in range(n_assemblies)])
    
    assembly_results[mouse_id] = {
        'W': W, 'H': H, 'composition': composition,
        'assembly_run_corr': assembly_run_corr,
        'subclass_names': subclass_names, 'run_speed': run_speed,
        'time': time_spont, 'reconstruction_error': nmf.reconstruction_err_
    }
    print(f"Mouse {mouse_id}: {n_cells} cells, {n_assemblies} assemblies, "
          f"recon error={nmf.reconstruction_err_:.1f}")

# ── Visualization ──
fig, axes = plt.subplots(len(MOUSE_IDS), 3, figsize=(20, 4 * len(MOUSE_IDS)))
if len(MOUSE_IDS) == 1:
    axes = axes[np.newaxis, :]

for i, mouse_id in enumerate(MOUSE_IDS):
    res = assembly_results[mouse_id]
    
    # Assembly activation timecourse (first 500 timepoints = 50 s)
    ax = axes[i, 0]
    t_plot = res['time'][:500]
    for a in range(min(5, res['W'].shape[1])):
        ax.plot(t_plot, res['W'][:500, a], alpha=0.7, label=f'Asm {a}')
    ax2 = ax.twinx()
    ax2.plot(t_plot, res['run_speed'][:500], 'k-', alpha=0.3, lw=0.5)
    ax2.set_ylabel('Run speed', fontsize=8)
    ax.set_title(f'Mouse {mouse_id}: Assembly Activations', fontweight='bold')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Activation')
    ax.legend(fontsize=6, loc='upper right')
    
    # Assembly–running correlation
    ax = axes[i, 1]
    colors = ['red' if c > 0.1 else 'blue' if c < -0.1 else 'gray'
              for c in res['assembly_run_corr']]
    ax.bar(range(len(res['assembly_run_corr'])), res['assembly_run_corr'], color=colors)
    ax.axhline(0, color='k', ls='--', alpha=0.4)
    ax.set_xlabel('Assembly')
    ax.set_ylabel('Correlation with running')
    ax.set_title(f'Mouse {mouse_id}: Assembly–Running Coupling', fontweight='bold')
    
    # Assembly composition (stacked bar)
    ax = axes[i, 2]
    all_types = sorted(set(t for comp in res['composition'].values() for t in comp))
    type_colors = [SUBCLASS_COLORS.get(t, '#cccccc') for t in all_types]
    bottoms = np.zeros(len(res['composition']))
    for j, t in enumerate(all_types):
        vals = [res['composition'][a].get(t, 0) for a in range(len(res['composition']))]
        ax.bar(range(len(vals)), vals, bottom=bottoms, color=type_colors[j],
               label=SUBCLASS_SHORT.get(t, t[-10:]))
        bottoms += vals
    ax.set_xlabel('Assembly')
    ax.set_ylabel('# top-10% cells')
    ax.set_title(f'Mouse {mouse_id}: Assembly Composition', fontweight='bold')
    ax.legend(fontsize=6, loc='upper right')

plt.tight_layout()
plt.show()

# ── Participation breadth by subclass ──
print("\n=== Cell participation breadth (# assemblies with above-median weight) ===")
for mouse_id in MOUSE_IDS:
    res = assembly_results[mouse_id]
    H = res['H']
    medians = np.median(H, axis=1, keepdims=True)
    participation = (H > medians).sum(axis=0)  # per cell, how many assemblies
    for sc in present_subclasses:
        mask = res['subclass_names'] == sc
        if mask.sum() > 0:
            print(f"  {mouse_id} {SUBCLASS_SHORT[sc]:10s}: "
                  f"mean={participation[mask].mean():.1f}, n={mask.sum()}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E5  Catch-Trial Expectation / Omission Signals (Zarr)
# ══════════════════════════════════════════════════════════════════════

catch_results = []
for mouse_id in MOUSE_IDS:
    z = zarr.open(f'{ZARR_DIR}/{mouse_id}_multimodal_data.zarr', 'r')
    subclass_names = z['transcriptomics/cell_type/subclass_name'][:]
    
    for sess in ['session_1', 'session_2', 'session_3']:
        catch = z[f'ophys/drifting_gratings/{sess}/stim_aligned_dff/Catch']
        dff_catch = catch['dff'][:]         # (27, 51, n_cells)
        time_catch = catch['time_relative'][:]  # (51,)
        running_catch = catch['running'][:]  # (27, 51, 2)
        
        # Trial info: which context block the catch occurred in
        stim_names = catch['trial_info/stim_name'][:]
        
        # Baseline: mean of −1 to 0 s (first 10 timepoints)
        baseline_idx = time_catch < 0
        stim_idx = (time_catch >= 0) & (time_catch <= 2.0)
        
        baseline_mean = dff_catch[:, baseline_idx, :].mean(axis=(0, 1))  # (n_cells,)
        stim_mean = dff_catch[:, stim_idx, :].mean(axis=(0, 1))          # (n_cells,)
        
        # Catch response index per cell
        catch_response = stim_mean - baseline_mean
        
        for ci, sc in enumerate(subclass_names):
            catch_results.append({
                'mouse_id': mouse_id, 'session': sess,
                'subclass': sc, 'catch_response': catch_response[ci],
                'cell_idx': ci
            })

catch_df = pd.DataFrame(catch_results)
catch_df['subclass_short'] = catch_df['subclass'].map(SUBCLASS_SHORT)

# ── Visualization ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Catch response distributions by subclass
ax = axes[0]
short_order = [SUBCLASS_SHORT[s] for s in present_subclasses]
short_pal = {SUBCLASS_SHORT[k]: v for k, v in SUBCLASS_COLORS.items()}
catch_avg = catch_df.groupby(['mouse_id', 'cell_idx', 'subclass_short'])['catch_response'].mean().reset_index()
sns.violinplot(data=catch_avg[catch_avg['subclass_short'].isin(short_order)],
               x='subclass_short', y='catch_response', order=short_order,
               palette=short_pal, cut=0, inner='box', ax=ax)
ax.axhline(0, color='k', ls='--', alpha=0.4)
ax.set_title('E5: Catch-Trial Response by Subclass', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('ΔF/F (stim − baseline)')
ax.tick_params(axis='x', rotation=45)

# 2. Mean PSTH for catch trials by subclass (session 1, mouse 778174)
ax = axes[1]
z = zarr.open(f'{ZARR_DIR}/778174_multimodal_data.zarr', 'r')
dff_c = z['ophys/drifting_gratings/session_1/stim_aligned_dff/Catch/dff'][:]
t_c = z['ophys/drifting_gratings/session_1/stim_aligned_dff/Catch/time_relative'][:]
sc_names = z['transcriptomics/cell_type/subclass_name'][:]
for sc in present_subclasses:
    mask = sc_names == sc
    if mask.sum() > 0:
        psth = dff_c[:, :, mask].mean(axis=(0, 2))
        ax.plot(t_c, psth, color=SUBCLASS_COLORS[sc], label=SUBCLASS_SHORT[sc], lw=1.5)
ax.axvline(0, color='k', ls='--', alpha=0.3, label='Stim onset')
ax.axhspan(-0.01, 0.01, color='gray', alpha=0.1)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Mean ΔF/F')
ax.set_title('E5: Catch-Trial PSTHs (778174, Session 1)', fontweight='bold')
ax.legend(fontsize=7)

# 3. Fraction of cells with significant catch responses
ax = axes[2]
sig_fractions = {}
for sc in present_subclasses:
    sc_data = catch_avg[catch_avg['subclass_short'] == SUBCLASS_SHORT[sc]]['catch_response']
    if len(sc_data) > 5:
        # Fraction significantly positive or negative (> 2 SD from 0)
        threshold = 2 * sc_data.std()
        frac_pos = (sc_data > threshold).mean()
        frac_neg = (sc_data < -threshold).mean()
        sig_fractions[SUBCLASS_SHORT[sc]] = {'positive': frac_pos, 'negative': frac_neg}

if sig_fractions:
    types = list(sig_fractions.keys())
    pos_vals = [sig_fractions[t]['positive'] for t in types]
    neg_vals = [sig_fractions[t]['negative'] for t in types]
    x = np.arange(len(types))
    ax.bar(x - 0.15, pos_vals, 0.3, color='firebrick', label='Activated', alpha=0.8)
    ax.bar(x + 0.15, neg_vals, 0.3, color='steelblue', label='Suppressed', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(types, rotation=45)
    ax.set_ylabel('Fraction of cells')
    ax.set_title('E5: Catch-Responsive Cells', fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# ── Statistics ──
print("Catch-trial response by subclass (Kruskal-Wallis):")
groups = [catch_avg.loc[catch_avg['subclass_short'] == SUBCLASS_SHORT[s], 'catch_response'].dropna().values
          for s in present_subclasses if (catch_avg['subclass_short'] == SUBCLASS_SHORT[s]).sum() >= 3]
if len(groups) >= 2:
    stat, p = kruskal(*groups)
    print(f"  H={stat:.2f}, p={p:.2e}")